In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

import os
from pathlib import Path
import sys

CWD = os.getcwd()
PARENT_DIR = os.path.dirname(CWD)
sys.path.append(PARENT_DIR)

In [2]:
from src.data import TextDataset
from torch.utils.data import DataLoader

from src.modules import QLoRA
import torch.optim as optim

MODEL_NAME = "Qwen/Qwen3.5-2B"
EPOCH_SIZE = 2000
EVAL_ITER = 2
LEARNING_RATE = 2e-4
ACCUMULATION_STEPS = 4

train_data = TextDataset(tokenizer_model=MODEL_NAME)
loader = DataLoader(train_data.data[train_data.data_sources[0]],
            batch_size=1,
            shuffle=True,
            collate_fn=lambda rows: rows[0],
        )
qwen_qlora = QLoRA(model_name=MODEL_NAME, rank=16, alpha=32, device=device)

c:\Users\taput\anaconda3\envs\chatbot-fraudster\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.
Loading weights: 100%|██████████| 320/320 [00:03<00:00, 87.72it/s] 


In [3]:
params    = [p for n, p in qwen_qlora.named_parameters() if p.requires_grad ]
optimizer = optim.AdamW(params=params, weight_decay=1e-2, lr=LEARNING_RATE)

# Train

In [4]:
qwen_qlora.train()
for step, batch in enumerate(loader, start=1):
    if step > EPOCH_SIZE:
        break

    batch = {
        key: value.unsqueeze(0).to("cuda:0")
        for key, value in batch.items()
    }

    loss = qwen_qlora(**batch).loss

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at step {step}: {loss.item()}")

    (loss / ACCUMULATION_STEPS).backward()
    if step % ACCUMULATION_STEPS == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    print(f"step {step}: loss={loss.item():.4f}")

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


step 1: loss=0.0504
step 2: loss=0.4684
step 3: loss=0.0831
step 4: loss=0.0766
step 5: loss=0.0131
step 6: loss=2.1118
step 7: loss=0.0255
step 8: loss=0.0823
step 9: loss=0.0482
step 10: loss=0.0588
step 11: loss=0.4089
step 12: loss=0.2859
step 13: loss=0.7866
step 14: loss=0.6896
step 15: loss=0.0464
step 16: loss=0.1692
step 17: loss=0.0035
step 18: loss=0.2468
step 19: loss=0.0217
step 20: loss=1.4586
step 21: loss=0.0186
step 22: loss=0.1356
step 23: loss=0.2423
step 24: loss=0.1326
step 25: loss=0.0505
step 26: loss=0.4135
step 27: loss=1.4688
step 28: loss=0.1156
step 29: loss=1.3499
step 30: loss=1.4909
step 31: loss=0.0926
step 32: loss=0.1380
step 33: loss=0.0478
step 34: loss=0.0189
step 35: loss=0.0037
step 36: loss=0.0165
step 37: loss=0.0068
step 38: loss=0.1155
step 39: loss=1.3776
step 40: loss=0.0047
step 41: loss=0.0182
step 42: loss=1.4443
step 43: loss=0.0153
step 44: loss=0.0209
step 45: loss=0.0154
step 46: loss=0.4810
step 47: loss=0.0231
step 48: loss=0.0349
s

AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.


In [ ]:
adapter_weights = {
    name: parameter.detach().cpu()
    for name, parameter in qwen_qlora.named_parameters()
    if name.endswith(("lora_a", "lora_b"))
}

torch.save(
    {
        "base_model": MODEL_NAME,
        "rank": 16,
        "alpha": 32,
        "weights": adapter_weights,
    },
    "../models/qwen3.5-2b-toolace-lora-1.pt",
)

In [ ]:
# Check which parameters are trainable.
trainable = [
    name for name, p in qwen_qlora.named_parameters()
    if p.requires_grad
]

print("Trainable tensors:", len(trainable))
print("First few:", trainable[:10])

assert all(
    name.endswith(("lora_a", "lora_b"))
    for name in trainable
), "A non-LoRA parameter is trainable"

In [ ]:
# Save it to Huggingface transformer version:

In [ ]:
import json
from pathlib import Path

import torch
from safetensors.torch import save_file

SOURCE = Path("../models/qwen3.5-2b-toolace-lora-1.pt")
OUTPUT = Path("../models/Qwen/qwen3.5-2b-toolace-adapter-1")

checkpoint = torch.load(SOURCE, map_location="cpu", weights_only=True)
assert checkpoint["base_model"] == "Qwen/Qwen3.5-2B"

weights = checkpoint["weights"]
converted = {}
targets = set()

for name, tensor in weights.items():
    if name.endswith(".lora_a"):
        module_name = name.removesuffix(".lora_a")
        suffix = "lora_A.weight"
    elif name.endswith(".lora_b"):
        module_name = name.removesuffix(".lora_b")
        suffix = "lora_B.weight"
    else:
        raise ValueError(f"Unexpected checkpoint key: {name}")

    # PEFT-style key: base_model.model.<original module path>.<A or B>
    new_name = f"base_model.model.{module_name}.{suffix}"
    converted[new_name] = tensor.detach().cpu().contiguous()
    targets.add(module_name.split(".")[-1])

assert len(converted) == len(weights)
assert len(converted) % 2 == 0

OUTPUT.mkdir(exist_ok=True)
save_file(converted, OUTPUT / "adapter_model.safetensors")

config = {
    "peft_type": "LORA",
    "task_type": "CAUSAL_LM",
    "base_model_name_or_path": checkpoint["base_model"],
    "r": checkpoint["rank"],
    "lora_alpha": checkpoint["alpha"],
    "lora_dropout": 0.0,
    "bias": "none",
    "target_modules": sorted(targets),
    "inference_mode": True,
}

(OUTPUT / "adapter_config.json").write_text(
    json.dumps(config, indent=2), encoding="utf-8"
)
print(f"Exported {len(converted) // 2} LoRA pairs")
print("Targets:", sorted(targets))

In [ ]:
# vllm serve Qwen/Qwen3.5-2B --enable-auto-tool-choice --tool-call-parser qwen3_coder --gpu-memoru-utilization 0.85
# bfcl generate --model qwen35-2b-base-FC --test-category simple_python,multi_turn --include-input-log